In [63]:
#DV: rating row[8]
import numpy as np
import matplotlib.pyplot as plt
import csv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score, recall_score
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

#import data from CSVs
data = []
files=["../2_Database/TopBeerData.csv","../2_Database/GoodBeerData.csv","../2_Database/RestBeerData.csv"]
for i in range (0,len(files)):
    with open(files[i],"r",encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # remove header
        for row in reader:
            data.append(row)

all_numeric_data = []
numeric_data = []
for row in data[1:]:  # ignore heading
    numeric_data.append([
        float(row[6]),  # 0 abv
        float(row[9]),   # 1 nr of raters
        float(row[12]),  # 2 malt
        float(row[13]),  # 3 hops
        float(row[14]),  # 4 chocolate
        float(row[15]),  # 5 german
        float(row[16]),  # 6 banana
        float(row[17]),  # 7 vanilla
        float(row[18]),  # 8 caramel
        float(row[19]),  # 9 yeast
        float(row[20]),  # 10 fruit
        float(row[21]),  # 11 whiskey
        float(row[22]),  # 12 rhum
        float(row[23]),  # 13 alcohol
        float(row[24]),  # 14 gluten
        float(row[25]),  # 15 barrel
        float(row[26]),  # 16 rye
        float(row[27]),  # 17 barley
        float(row[28]),  # 18 cocoa
        float(row[29]),  # 19 small/lower
        float(row[30]),  # 20 big/upper
        float(row[31] + row[32] + row[40] + row[49]),  # 21 Altbier and kölsch and Barleywine and Other
        float(row[33] + row[34] + row[35]+ row[36]),  # 22 Blonde and Duppel and Quadrupel and Tripel
        float(row[37] + row[38]),  # 23 Cider and Fruit
        float(row[39]),  # 24 IPA
        float(row[41]),  # 25 Lager
        float(row[42]),  # 26 Non-Alcoholic
        float(row[43]),  # 27 Pilsner
        float(row[44]),  # 28 Porter
        float(row[45]),  # 29 Wheat
        float(row[46]),  # 30 Sour
        float(row[47]),  # 31 Stout
        float(row[48])  # 32 Ale
    ])
numeric_data = np.array(numeric_data)
all_numeric_data.append(numeric_data)
combined = np.vstack(all_numeric_data)
print(combined.shape)

#IDVs + DV def
X = combined[:, :20]
#y = combined[:, 21:33]
y = np.argmax(combined[:, 22:31], axis=1) # ignore 1st and last feature, because those are misleadingly "colorful" groups
#print(X)
print(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# -------------------------
# MODEL
# -------------------------

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
# multiclass prediction
y_pred = model.predict(X_test)
# probabilities
y_prob = model.predict_proba(X_test)

# -------------------------
# METRICS
# -------------------------
print("\nConfusionMatrix\n", confusion_matrix(y_test, y_pred))
print("\nAccuracy", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred,average="weighted"))
print("Recall:", recall_score(y_test, y_pred,average="weighted"))
print("F1-score:", (2*precision_score(y_test, y_pred,average="weighted")*recall_score(y_test, y_pred,average="weighted"))/(precision_score(y_test, y_pred,average="weighted")+recall_score(y_test, y_pred,average="weighted")))
print("ClassificationReport:\n",classification_report(y_test, y_pred))

(2302, 33)
[0 2 0 ... 0 0 0]

ConfusionMatrix
 [[167  21  41  16   0   8   4   5  11]
 [ 16  25   8   5   0   1   2   2  11]
 [ 38   7  84   8   1   2   3   3   4]
 [ 14   6  11   9   0   4   2   4   3]
 [  0   0   1   1   9   0   0   0   0]
 [  8   3   4   6   0   7   0   1   0]
 [ 20   2   5   0   0   1   2   0   0]
 [ 17   4   3   6   0   1   0   2   1]
 [ 18   6   3   6   0   1   0   1   6]]

Accuracy 0.45007235890014474
Precision: 0.4298169295422315
Recall: 0.45007235890014474
F1-score: 0.4397114998791314
ClassificationReport:
               precision    recall  f1-score   support

           0       0.56      0.61      0.58       273
           1       0.34      0.36      0.35        70
           2       0.53      0.56      0.54       150
           3       0.16      0.17      0.16        53
           4       0.90      0.82      0.86        11
           5       0.28      0.24      0.26        29
           6       0.15      0.07      0.09        30
           7       0.11     

In [64]:
import pandas
import pandas as pd

fi = pd.DataFrame({
    "feature": range(X.shape[1]),
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(fi)

    feature  importance
1         1    0.408740
0         0    0.369034
3         3    0.033524
10       10    0.024917
13       13    0.018410
2         2    0.018366
4         4    0.018228
19       19    0.016240
9         9    0.014562
6         6    0.013466
7         7    0.012218
15       15    0.011741
5         5    0.010014
18       18    0.006738
12       12    0.006535
17       17    0.005830
8         8    0.005144
11       11    0.003136
16       16    0.002313
14       14    0.000843


In [65]:
top_idx = np.argsort(model.feature_importances_)[-14:]   # legnagyobb importance
top_idx = top_idx[::-1]                   # csökkenő sorrend
print("Top feature indexek:", top_idx)

X_top = X[:, top_idx]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_top,
    y,
    test_size=0.3, 
    random_state=42, 
    stratify=y
)

model2 = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)

# -------------------------
# METRICS
# -------------------------
print("\nConfusionMatrix\n", confusion_matrix(y_test2, y_pred2))
print("\nAccuracy", accuracy_score(y_test2, y_pred2))
print("Precision:", precision_score(y_test2, y_pred2,average="weighted"))
print("Recall:", recall_score(y_test2, y_pred2,average="weighted"))
print("F1-score:", (2*precision_score(y_test2, y_pred2,average="weighted")*recall_score(y_test2, y_pred2,average="weighted"))/(precision_score(y_test2, y_pred2,average="weighted")+recall_score(y_test2, y_pred2,average="weighted")))
print("ClassificationReport:\n",classification_report(y_test2, y_pred2))


Top feature indexek: [ 1  0  3 10 13  2  4 19  9  6  7 15  5 18]

ConfusionMatrix
 [[171  16  43  13   0   8   7   5  10]
 [ 17  23   8   5   0   2   2   2  11]
 [ 37   9  84   6   1   2   3   4   4]
 [ 16   6   8   9   0   4   2   5   3]
 [  0   0   1   1   9   0   0   0   0]
 [  7   3   5   6   0   7   0   1   0]
 [ 21   1   5   0   0   1   2   0   0]
 [ 16   3   4   6   0   1   0   2   2]
 [ 18   7   3   6   0   1   0   1   5]]

Accuracy 0.4515195369030391
Precision: 0.4282120972099245
Recall: 0.4515195369030391
F1-score: 0.4395570656577692
ClassificationReport:
               precision    recall  f1-score   support

           0       0.56      0.63      0.59       273
           1       0.34      0.33      0.33        70
           2       0.52      0.56      0.54       150
           3       0.17      0.17      0.17        53
           4       0.90      0.82      0.86        11
           5       0.27      0.24      0.25        29
           6       0.12      0.07      0.09     

In [66]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# -------------------------
# TOP FEATURES
# -------------------------
top_idx = np.argsort(model.feature_importances_)[-14:]
top_idx = top_idx[::-1]
print("Top feature indexek:", top_idx)
X_top = X[:, top_idx]
# -------------------------
# TRAIN TEST SPLIT
# -------------------------

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_top,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# -------------------------
# XGBOOST MODEL
# -------------------------
model2 = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(np.unique(y)),
    random_state=42,
    n_jobs=-1
)

# -------------------------
# TRAIN
# -------------------------
model2.fit(X_train2, y_train2)

# -------------------------
# PREDICTION
# -------------------------
y_pred2 = model2.predict(X_test2)

# -------------------------
# METRICS
# -------------------------
print("\nConfusionMatrix\n",confusion_matrix(y_test2, y_pred2))
print("\nAccuracy",accuracy_score(y_test2, y_pred2))
print("Precision:",precision_score(y_test2,y_pred2,average="weighted"))
print("Recall:", recall_score(y_test2,y_pred2,average="weighted"))
print("F1-score:",f1_score(y_test2,y_pred2,average="weighted"))
print("ClassificationReport:\n",classification_report(y_test2,y_pred2))

Top feature indexek: [ 1  0  3 10 13  2  4 19  9  6  7 15  5 18]

ConfusionMatrix
 [[202  11  33  15   0   3   2   3   4]
 [ 29  26   4   5   0   0   0   0   6]
 [ 54   4  82   6   0   1   0   1   2]
 [ 24   2   5  12   1   4   1   3   1]
 [  0   0   1   0  10   0   0   0   0]
 [ 11   2   2  10   0   3   0   1   0]
 [ 22   2   3   0   0   1   1   0   1]
 [ 23   1   1   5   0   1   0   1   2]
 [ 23   8   1   2   0   1   0   4   2]]

Accuracy 0.4905933429811867
Precision: 0.44900079075262206
Recall: 0.4905933429811867
F1-score: 0.45556392854387473
ClassificationReport:
               precision    recall  f1-score   support

           0       0.52      0.74      0.61       273
           1       0.46      0.37      0.41        70
           2       0.62      0.55      0.58       150
           3       0.22      0.23      0.22        53
           4       0.91      0.91      0.91        11
           5       0.21      0.10      0.14        29
           6       0.25      0.03      0.06   